# EXAONE-Deep-7.8B LoRA 재학습 v2 — 민원 답변 품질 개선

## 목적
기존 LoRA 학습의 근본 문제를 해결하고 민원 답변 품질을 대폭 개선합니다.

### 기존 문제점
| 문제 | 원인 | 해결 방안 |
|------|------|----------|
| EOS 미생성 (p50 latency 27s) | `[|endofturn|]` 학습 누락 | output 끝에 EOS 토큰 포함 + DataCollatorForCompletionOnlyLM |
| SacreBLEU 0.53 | 출력 스타일 불일치 | chat template 정확 적용 + 답변 품질 필터링 |
| ROUGE-L 4.20 | PII 마스킹 잔여 | PII 통일 + 고밀도 PII 샘플 제거 |
| length_ratio 0.63 | 학습 데이터 답변 길이 부족 | 30자 미만 제거 + 3 epoch 학습 |

### 핵심 개선사항
1. **EOS 토큰 학습**: `[|endofturn|]`이 포함된 chat template 텍스트로 직접 학습
2. **CompletionOnly Loss**: `[|assistant|]` 이후 응답 부분만 loss 계산
3. **데이터 품질 강화**: PII 통일, 짧은 답변 제거, 반복 패턴 제거
4. **3 epoch 학습**: 기존 1 epoch에서 증가하여 충분한 패턴 학습

### 환경 요구사항
- **GPU**: Google Colab A100 (40GB) 또는 L4 (24GB)
- **런타임**: Python 3, GPU 가속
- **예상 학습 시간**: A100 기준 약 2-3시간 (3 epoch)

### 데이터 흐름
- 데이터 전처리는 **로컬에서 완료** (`reconstruct_data_v2.py`)
- Colab에서는 **git clone → 학습 → 업로드**만 수행
- 학습 데이터: `data/processed/v2_train.jsonl` (EXAONE chat template 포맷, `text` 필드)

## Cell 1. 패키지 설치

transformers 4.44~4.49 범위로 고정하여 EXAONE 호환성을 보장합니다.

In [ ]:
%%capture
# === 패키지 설치 (transformers 4.44~4.49 고정) ===
!pip install "transformers>=4.44,<4.50" peft trl bitsandbytes datasets
!pip install wandb accelerate sacrebleu rouge_score bert_score
!pip install huggingface_hub tqdm numpy

print("패키지 설치 완료")

## Cell 2. 환경 초기화

W&B, HuggingFace 로그인 및 프로젝트 클론. EXAONE revision을 고정합니다.
데이터는 git clone으로 가져온 `data/processed/v2_*.jsonl`을 바로 사용합니다.

In [ ]:
import os
import torch

# ─── 상수 정의 ───────────────────────────────────────────────────────
EXAONE_REVISION = "17b70148e344"
BASE_MODEL_ID = "LGAI-EXAONE/EXAONE-Deep-7.8B"
HF_REPO_ID = "umyunsang/civil-complaint-exaone-lora-v2"
WANDB_PROJECT = "civil-complaint-retrain-v2"
SEED = 42
MAX_SEQ_LENGTH = 2048

# ─── 프로젝트 클론 ────────────────────────────────────────────────────
PROJECT_ROOT = "/content/GovOn"
if not os.path.exists(PROJECT_ROOT):
    !git clone https://github.com/GovOn-Org/GovOn.git {PROJECT_ROOT}
    print(f"프로젝트 클론 완료: {PROJECT_ROOT}")
else:
    # 이미 클론된 경우 최신 pull
    !cd {PROJECT_ROOT} && git pull
    print(f"프로젝트 이미 존재, pull 완료: {PROJECT_ROOT}")

# ─── 데이터 경로 확인 ─────────────────────────────────────────────────
TRAIN_PATH = os.path.join(PROJECT_ROOT, "data/processed/v2_train.jsonl")
VAL_PATH = os.path.join(PROJECT_ROOT, "data/processed/v2_val.jsonl")
TEST_PATH = os.path.join(PROJECT_ROOT, "data/processed/v2_test.jsonl")

for path in [TRAIN_PATH, VAL_PATH, TEST_PATH]:
    assert os.path.exists(path), f"데이터 파일 없음: {path}"
print("데이터 파일 확인 완료")

# ─── 작업 디렉토리 ────────────────────────────────────────────────────
CHECKPOINT_DIR = "/content/checkpoints/retrain-v2"
RESULTS_DIR = "/content/results"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

# ─── W&B 로그인 ──────────────────────────────────────────────────────
import wandb
wandb.login(key="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")
print("W&B 로그인 완료")

# ─── HuggingFace 로그인 ──────────────────────────────────────────────
from huggingface_hub import login as hf_login
try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
    hf_login(token=hf_token)
    print("HuggingFace 로그인 완료 (Colab Secrets)")
except Exception:
    print("[안내] HuggingFace 토큰이 필요합니다. Colab Secrets에 HF_TOKEN을 등록하세요.")
    # hf_login(token="hf_YOUR_TOKEN_HERE")

# ─── GPU 확인 ─────────────────────────────────────────────────────────
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"
gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0
print(f"GPU: {gpu_name}, VRAM: {gpu_mem:.1f} GB")
assert torch.cuda.is_available(), "GPU가 필요합니다!"

# ─── transformers 버전 확인 ───────────────────────────────────────────
import transformers
print(f"transformers: {transformers.__version__}")
assert transformers.__version__.startswith("4.4"), \
    f"transformers 4.44~4.49 필요, 현재: {transformers.__version__}"

## Cell 3. 데이터 로드 + 통계 확인

로컬에서 전처리 완료된 v2 데이터를 로드합니다.
각 레코드는 `{"text": "...", "category": "...", "id": "..."}` 형식이며,
`text` 필드에 EXAONE chat template이 적용된 완성된 학습 텍스트가 들어있습니다.

In [ ]:
import json
import numpy as np
from collections import Counter
from datasets import Dataset

# ─── JSONL 로드 함수 ──────────────────────────────────────────────────
def load_jsonl(path):
    records = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records

train_data = load_jsonl(TRAIN_PATH)
val_data = load_jsonl(VAL_PATH)
test_data = load_jsonl(TEST_PATH)

train_ds = Dataset.from_list(train_data)
val_ds = Dataset.from_list(val_data)

# ─── 통계 출력 ────────────────────────────────────────────────────────
print("=" * 60)
print("  v2 데이터셋 통계")
print("=" * 60)
print(f"Train: {len(train_data):,}")
print(f"Val:   {len(val_data):,}")
print(f"Test:  {len(test_data):,}")
print(f"Total: {len(train_data) + len(val_data) + len(test_data):,}")

# 카테고리 분포
print("\n--- 카테고리 분포 (Train) ---")
cat_dist = Counter(r["category"] for r in train_data)
for cat, cnt in sorted(cat_dist.items(), key=lambda x: -x[1]):
    pct = cnt / len(train_data) * 100
    bar = "#" * int(pct)
    print(f"  {cat:<6s}: {cnt:>5d} ({pct:>5.1f}%) {bar}")

# text 길이 통계
print("\n--- text 필드 길이 ---")
for name, data in [("Train", train_data), ("Val", val_data), ("Test", test_data)]:
    lens = [len(r["text"]) for r in data]
    print(f"  {name}: 평균 {np.mean(lens):.0f}, 중앙값 {np.median(lens):.0f}, "
          f"최소 {min(lens)}, 최대 {max(lens)}")

# 포맷 검증
print("\n--- 포맷 검증 (첫 번째 샘플) ---")
sample = train_data[0]
print(f"ID: {sample['id']}")
print(f"Category: {sample['category']}")
print(f"Text 시작: {sample['text'][:200]}")
print(f"...")
print(f"Text 끝: {sample['text'][-80:]}")
assert sample["text"].endswith("[|endofturn|]"), "text가 [|endofturn|]으로 끝나지 않습니다!"
assert "[|assistant|]" in sample["text"], "text에 [|assistant|]가 없습니다!"
print("\n포맷 검증 통과")

## Cell 4. 모델 로딩 (QLoRA 4-bit)

EXAONE-Deep-7.8B를 QLoRA NF4로 로드합니다.
- revision: `17b70148e344` (학습 호환 코드 고정)
- `trust_remote_code=True` 필수
- EXAONE 호환성 패치 적용

In [ ]:
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    set_seed,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

set_seed(SEED)

# ─── 토크나이저 로드 ──────────────────────────────────────────────────
print("토크나이저 로딩...")
tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL_ID,
    revision=EXAONE_REVISION,
    trust_remote_code=True,
)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print(f"EOS 토큰: '{tokenizer.eos_token}' (id={tokenizer.eos_token_id})")
print(f"PAD 토큰: '{tokenizer.pad_token}' (id={tokenizer.pad_token_id})")

# ─── 4-bit 양자화 설정 ───────────────────────────────────────────────
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# ─── 모델 로드 ────────────────────────────────────────────────────────
print("\n모델 로딩 중... (수 분 소요)")
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    revision=EXAONE_REVISION,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
)

# ─── EXAONE 호환성 패치 ──────────────────────────────────────────────
try:
    model.get_input_embeddings()
except (NotImplementedError, AttributeError):
    model.get_input_embeddings = lambda: model.transformer.wte
    print("[패치] get_input_embeddings")

try:
    model.get_output_embeddings()
except (NotImplementedError, AttributeError):
    model.get_output_embeddings = lambda: model.lm_head
    print("[패치] get_output_embeddings")

# ─── kbit 학습 준비 ──────────────────────────────────────────────────
model = prepare_model_for_kbit_training(model)

allocated = torch.cuda.memory_allocated() / 1024**3
reserved = torch.cuda.memory_reserved() / 1024**3
print(f"\nGPU 메모리: {allocated:.1f}GB allocated, {reserved:.1f}GB reserved")
print("모델 로딩 완료!")

## Cell 5. LoRA 설정 + 학습 설정

- LoRA: r=16, alpha=32, target_modules=[q,k,v,o,gate,up,down]_proj
- DataCollatorForCompletionOnlyLM: `[|assistant|]` 이후 응답 부분만 loss 계산
- v2 데이터는 `text` 필드에 완성된 chat template이 있으므로 `dataset_text_field="text"` 사용
- epochs=3, lr=2e-4, cosine scheduler, warmup_ratio=0.03

In [ ]:
from trl import SFTTrainer, DataCollatorForCompletionOnlyLM

# =====================================================================
#  LoRA 설정
# =====================================================================

LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
]

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=TARGET_MODULES,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# =====================================================================
#  DataCollatorForCompletionOnlyLM 설정
# =====================================================================

RESPONSE_TEMPLATE = "[|assistant|]"

data_collator = DataCollatorForCompletionOnlyLM(
    response_template=RESPONSE_TEMPLATE,
    tokenizer=tokenizer,
)
print(f"DataCollator response_template: '{RESPONSE_TEMPLATE}'")
print(f"  -> [|assistant|] 이후 응답 + [|endofturn|] 부분만 loss 계산")

# =====================================================================
#  학습 하이퍼파라미터
# =====================================================================

NUM_EPOCHS = 3
LEARNING_RATE = 2e-4
BATCH_SIZE = 2
GRAD_ACCUM = 8
WARMUP_RATIO = 0.03

effective_batch = BATCH_SIZE * GRAD_ACCUM
total_steps_approx = (len(train_ds) // effective_batch) * NUM_EPOCHS

print(f"\n--- 학습 하이퍼파라미터 ---")
print(f"  Epochs: {NUM_EPOCHS}")
print(f"  Learning Rate: {LEARNING_RATE}")
print(f"  Batch Size: {BATCH_SIZE} x Grad Accum {GRAD_ACCUM} = Effective {effective_batch}")
print(f"  Warmup Ratio: {WARMUP_RATIO}")
print(f"  Max Seq Length: {MAX_SEQ_LENGTH}")
print(f"  예상 총 steps: ~{total_steps_approx}")

# ─── W&B run 초기화 ───────────────────────────────────────────────────
run = wandb.init(
    project=WANDB_PROJECT,
    name="retrain-v2-lora-r16-3ep",
    tags=["qlora", "exaone", "retrain-v2", "completion-only", "eos-fix"],
    config={
        "base_model": BASE_MODEL_ID,
        "revision": EXAONE_REVISION,
        "lora_r": LORA_R,
        "lora_alpha": LORA_ALPHA,
        "lora_dropout": LORA_DROPOUT,
        "learning_rate": LEARNING_RATE,
        "num_epochs": NUM_EPOCHS,
        "batch_size": BATCH_SIZE,
        "grad_accum": GRAD_ACCUM,
        "effective_batch_size": effective_batch,
        "warmup_ratio": WARMUP_RATIO,
        "max_seq_length": MAX_SEQ_LENGTH,
        "response_template": RESPONSE_TEMPLATE,
        "data_collator": "DataCollatorForCompletionOnlyLM",
        "data_format": "v2_chat_template_text_field",
        "train_samples": len(train_ds),
        "val_samples": len(val_ds),
    },
    reinit=True,
)

# ─── TrainingArguments ────────────────────────────────────────────────
training_args = TrainingArguments(
    output_dir=CHECKPOINT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    eval_accumulation_steps=10,
    learning_rate=LEARNING_RATE,
    num_train_epochs=NUM_EPOCHS,
    lr_scheduler_type="cosine",
    warmup_ratio=WARMUP_RATIO,
    weight_decay=0.01,
    max_grad_norm=1.0,
    bf16=True,
    tf32=True,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="wandb",
    run_name="retrain-v2-lora-r16-3ep",
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    seed=SEED,
    dataloader_num_workers=2,
)

print("\n학습 설정 완료!")

## Cell 6. SFTTrainer 학습 실행

v2 데이터의 `text` 필드를 `dataset_text_field`로 직접 사용합니다.
A100 기준 약 2-3시간 소요됩니다.

In [ ]:
# =====================================================================
#  SFTTrainer 구성 및 학습 실행
# =====================================================================

trainer = SFTTrainer(
    model=model,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    dataset_text_field="text",
    data_collator=data_collator,
    max_seq_length=MAX_SEQ_LENGTH,
    tokenizer=tokenizer,
    args=training_args,
)

print("SFTTrainer 구성 완료. 학습을 시작합니다...")
print(f"  Train samples: {len(train_ds)}")
print(f"  Val samples: {len(val_ds)}")
print(f"  Epochs: {NUM_EPOCHS}")
print(f"  Effective batch size: {BATCH_SIZE * GRAD_ACCUM}")

# 학습 실행
train_result = trainer.train()

# 학습 결과 출력
train_metrics = train_result.metrics
print(f"\n{'='*60}")
print(f"  학습 완료!")
print(f"{'='*60}")
print(f"  Train Loss: {train_metrics.get('train_loss', 'N/A'):.4f}")
print(f"  Train Runtime: {train_metrics.get('train_runtime', 0)/60:.1f}분")
print(f"  Train Samples/sec: {train_metrics.get('train_samples_per_second', 0):.2f}")

# 최종 모델 저장
FINAL_ADAPTER_DIR = os.path.join(CHECKPOINT_DIR, "final-adapter")
trainer.save_model(FINAL_ADAPTER_DIR)
tokenizer.save_pretrained(FINAL_ADAPTER_DIR)
print(f"\nLoRA 어댑터 저장: {FINAL_ADAPTER_DIR}")

## Cell 7. Sanity Check (생성 테스트)

학습된 모델로 샘플 민원에 대해 답변을 생성하여
EOS 토큰 생성 여부, 답변 품질, 응답 길이를 확인합니다.

In [ ]:
# =====================================================================
#  Sanity Check: 생성 테스트
# =====================================================================

model.eval()

SYSTEM_MESSAGE = "당신은 지자체 민원 담당 공무원을 돕는 AI 어시스턴트입니다."
INSTRUCTION = "다음 민원에 대해 공손하고 명확한 답변을 작성하세요."

# test set에서 5개 샘플 추출
test_samples = test_data[:5]

print("=" * 70)
print("  Sanity Check: 생성 테스트 (5 샘플)")
print("=" * 70)

eos_generated_count = 0
generation_lengths = []

for i, sample in enumerate(test_samples):
    # text에서 user 입력 부분 추출
    text = sample["text"]
    assistant_marker = "[|assistant|]"
    assistant_idx = text.find(assistant_marker)
    ref_answer = text[assistant_idx + len(assistant_marker):].replace("[|endofturn|]", "").strip()

    # user 부분 추출 (system과 user 사이)
    user_start = text.find("[|user|]") + len("[|user|]")
    user_end = text.find("[|endofturn|]", user_start)
    user_content = text[user_start:user_end].strip()

    messages = [
        {"role": "system", "content": SYSTEM_MESSAGE},
        {"role": "user", "content": user_content},
    ]

    input_ids = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to(model.device)

    if isinstance(input_ids, dict):
        gen_input = input_ids
        input_len = gen_input["input_ids"].shape[1]
    else:
        gen_input = {"input_ids": input_ids}
        input_len = input_ids.shape[1]

    with torch.no_grad():
        output_ids = model.generate(
            **gen_input,
            max_new_tokens=512,
            do_sample=False,
            temperature=1.0,
            eos_token_id=tokenizer.eos_token_id,
        )

    generated_ids = output_ids[0][input_len:]
    generated_text = tokenizer.decode(generated_ids, skip_special_tokens=False)
    generated_clean = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

    eos_in_output = tokenizer.eos_token_id in generated_ids.tolist()
    if eos_in_output:
        eos_generated_count += 1

    generation_lengths.append(len(generated_clean))

    print(f"\n--- [{i+1}] 카테고리: {sample['category']} ---")
    print(f"  생성 ({len(generated_clean)}자): {generated_clean[:200]}...")
    print(f"  참조 ({len(ref_answer)}자): {ref_answer[:200]}...")
    print(f"  EOS 생성: {'YES' if eos_in_output else 'NO'}")
    print(f"  생성 토큰 수: {len(generated_ids)}")

# 요약
print(f"\n{'='*70}")
print(f"  Sanity Check 요약")
print(f"{'='*70}")
print(f"  EOS 생성 비율: {eos_generated_count}/{len(test_samples)} "
      f"({eos_generated_count/len(test_samples)*100:.0f}%)")
print(f"  평균 생성 길이: {np.mean(generation_lengths):.0f}자")
print(f"  생성 길이 범위: {min(generation_lengths)} ~ {max(generation_lengths)}자")

if eos_generated_count == len(test_samples):
    print("\n  [PASS] 모든 샘플에서 EOS 토큰이 생성되었습니다!")
elif eos_generated_count > 0:
    print(f"\n  [PARTIAL] 일부 샘플에서 EOS 미생성. 추가 학습 고려.")
else:
    print(f"\n  [FAIL] EOS 토큰이 전혀 생성되지 않았습니다. 학습 설정 확인 필요.")

## Cell 8. HuggingFace Hub 업로드

학습된 LoRA 어댑터를 `umyunsang/civil-complaint-exaone-lora-v2`에 업로드합니다.

In [ ]:
# =====================================================================
#  HuggingFace Hub 업로드
# =====================================================================

from huggingface_hub import HfApi

api = HfApi()

print(f"LoRA 어댑터 업로드: {HF_REPO_ID}")
print(f"소스 디렉토리: {FINAL_ADAPTER_DIR}")

adapter_files = os.listdir(FINAL_ADAPTER_DIR)
print(f"업로드 파일: {adapter_files}")

try:
    api.upload_folder(
        folder_path=FINAL_ADAPTER_DIR,
        repo_id=HF_REPO_ID,
        repo_type="model",
        commit_message=(
            f"Upload retrain-v2 LoRA adapter: "
            f"r={LORA_R}, lr={LEARNING_RATE}, {NUM_EPOCHS}ep, "
            f"CompletionOnlyLM, EOS fix, v2 chat template data"
        ),
    )
    print(f"\n업로드 완료: https://huggingface.co/{HF_REPO_ID}")
except Exception as e:
    print(f"\n[오류] 업로드 실패: {e}")
    print("HuggingFace 토큰 확인 후 수동 업로드하세요:")
    print(f"  huggingface-cli upload {HF_REPO_ID} {FINAL_ADAPTER_DIR}")

## Cell 9. 결과 저장 + 다운로드

학습 결과를 JSON으로 저장하고 Colab에서 자동 다운로드합니다.

In [ ]:
# =====================================================================
#  결과 JSON 저장 및 자동 다운로드
# =====================================================================

from datetime import datetime

results = {
    "experiment": "retrain-v2-lora-r16-3ep",
    "timestamp": datetime.now().isoformat(),
    "base_model": BASE_MODEL_ID,
    "revision": EXAONE_REVISION,
    "hf_repo": HF_REPO_ID,
    "config": {
        "lora_r": LORA_R,
        "lora_alpha": LORA_ALPHA,
        "lora_dropout": LORA_DROPOUT,
        "target_modules": TARGET_MODULES,
        "learning_rate": LEARNING_RATE,
        "num_epochs": NUM_EPOCHS,
        "batch_size": BATCH_SIZE,
        "grad_accum": GRAD_ACCUM,
        "effective_batch_size": BATCH_SIZE * GRAD_ACCUM,
        "warmup_ratio": WARMUP_RATIO,
        "max_seq_length": MAX_SEQ_LENGTH,
        "lr_scheduler": "cosine",
        "optimizer": "paged_adamw_8bit",
        "response_template": RESPONSE_TEMPLATE,
        "data_collator": "DataCollatorForCompletionOnlyLM",
        "data_format": "v2_chat_template_text_field",
    },
    "data": {
        "train_samples": len(train_data),
        "val_samples": len(val_data),
        "test_samples": len(test_data),
        "data_source": "git clone (data/processed/v2_*.jsonl)",
    },
    "training_metrics": {
        "train_loss": train_metrics.get("train_loss"),
        "train_runtime_min": train_metrics.get("train_runtime", 0) / 60,
        "train_samples_per_sec": train_metrics.get("train_samples_per_second"),
    },
    "sanity_check": {
        "eos_generation_rate": eos_generated_count / len(test_samples),
        "avg_generation_length": float(np.mean(generation_lengths)),
    },
    "improvements_over_v1": [
        "데이터 전처리를 로컬에서 수행, Colab에서는 학습만",
        "text 필드에 완성된 EXAONE chat template 직접 저장",
        "EOS 토큰 [|endofturn|] 학습 포함",
        "DataCollatorForCompletionOnlyLM으로 응답 부분만 loss 계산",
        "PII 마스킹 v2 (특수문자 제거, 밀도 50% 초과 제거)",
        "품질 필터링 강화 (Jaccard > 0.8, 반복 패턴, 30자 미만 제거)",
        "3 epoch 학습",
    ],
    "wandb_run_id": run.id if run else None,
    "wandb_run_url": run.url if run else None,
}

results_path = os.path.join(RESULTS_DIR, "retrain_v2_results.json")
with open(results_path, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False, default=str)
print(f"결과 저장: {results_path}")

# W&B 종료
wandb.finish()
print("W&B run 종료")

# Colab 자동 다운로드
try:
    from google.colab import files
    files.download(results_path)
    print(f"\n자동 다운로드 시작: {os.path.basename(results_path)}")
except ImportError:
    print(f"\n[안내] Colab 환경이 아닙니다. 수동으로 다운로드하세요:")
    print(f"  {results_path}")

# 최종 요약
print(f"\n{'='*70}")
print(f"  EXAONE-Deep-7.8B LoRA 재학습 v2 완료!")
print(f"{'='*70}")
print(f"  모델: {BASE_MODEL_ID} (rev: {EXAONE_REVISION})")
print(f"  LoRA: r={LORA_R}, alpha={LORA_ALPHA}")
print(f"  학습: {NUM_EPOCHS} epochs, lr={LEARNING_RATE}")
print(f"  Train Loss: {train_metrics.get('train_loss', 'N/A'):.4f}")
print(f"  EOS 생성률: {eos_generated_count}/{len(test_samples)}")
print(f"  HuggingFace: https://huggingface.co/{HF_REPO_ID}")
print(f"  W&B: {run.url if run else 'N/A'}")
print(f"{'='*70}")